# C02 — Engenharia de Prompt e Saída Controlada

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)

**Autor:** Anderson Corrêa

**Projeto:** Letra da Lei

Este notebook cobre o **ponto 2 da rubrica** (engenharia de prompt + saída controlada).

O objetivo é mostrar o desenvolvimento do prompt usado por `answer_question`
(`c05_rag_pipeline.ipynb`). Cada etapa tem um **critério de qualidade**: *o modelo
preenche corretamente o array `citations` e define `abstained` de forma coerente com o
contexto fornecido?*

1. **Técnica 1 — zero-shot**: instrução simples, sem exemplo, sem saída controlada.
2. **Técnica 2 — JSON não-guiado**: `format="json"` do Ollama garante JSON válido, mas sem
   contrato de chaves.
3. **Técnica 3 — few-shot + saída controlada por schema**: `SYSTEM_PROMPT` traz um exemplo (few-shot) *e* a chamada usa `format=<json schema>` do Ollama, que restringe a
   saída à forma exata esperada por `parse_answer`.
4. **Técnica 4 — chain-of-thought estruturado**: campo `raciocinio` gerado *antes* do
   veredito, e aplicado ao conflito aparente de normas (art. 121 vs. art. 123).
5. `parse_answer` — parsing tolerante para o caso de saída não-parseável.

## Setup

Importam-se a API pública de `direito_dados.generation` (a mesma usada pelos testes do
projeto) e dois dispositivos do Código Penal (via `direito_dados.corpus`) para servir
de contexto nos prompts. 

*Nota*: construção de índice vetorial completo é demonstrado em `c03_embeddings_busca.ipynb` e `c05_rag_pipeline.ipynb`.

In [ ]:
import sys
from pathlib import Path

# make sure the local package is importable even if the kernel starts in a different cwd
_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.generation.llm import OllamaClient, ollama_available, ollama_has_model
from direito_dados.generation.prompt import SYSTEM_PROMPT, build_user_prompt
from direito_dados.generation.parse import parse_answer
from direito_dados.retrieval.index import Result

MODEL = "llama3.1:8b"
assert ollama_available(), "Ollama precisa estar rodando em localhost:11434"
assert ollama_has_model(MODEL), f"Modelo {MODEL} precisa estar baixado (`ollama pull {MODEL}`)"
llm = OllamaClient(model=MODEL)
# OllamaClient defaults to json_format=True (the production pipeline always expects JSON) —
# for Technique 1 (a "real" zero-shot: free prose, no format restriction at all) we need an
# instance with json_format=False.
llm_prose = OllamaClient(model=MODEL, json_format=False)
print(f"Ollama disponível, modelo {MODEL} pronto.")

In [2]:
corpus = load_corpus("data/raw", specs=[NORMS["CP"]])
cp = corpus.norms[0]
art121 = next(a for a in cp.articles if a.number == "121")
art155 = next(a for a in cp.articles if a.number == "155")

results = [
    Result(id="CP:art121", text=art121.text, citation=art121.citation, score=1.0,
           metadata={"status": art121.status.value, "citation": art121.citation}),
    Result(id="CP:art155", text=art155.text, citation=art155.citation, score=1.0,
           metadata={"status": art155.status.value, "citation": art155.citation}),
]
QUESTION = "Qual a pena para quem mata alguém?"
print(f"Contexto: {[r.id for r in results]}")
print(f"Pergunta: {QUESTION!r}")

Contexto: ['CP:art121', 'CP:art155']
Pergunta: 'Qual a pena para quem mata alguém?'


## Técnica 1 — Zero-shot (instrução simples, sem exemplo, sem saída controlada)

A primeira iteração foi pedir a resposta em linguagem natural, sem system prompt, sem exemplo e sem `format`. É o baseline contra o qual
as técnicas seguintes serão comparadas.

**Critério de qualidade:** o resultado precisa ser JSON parseável com `citations`
preenchido corretamente. Zero-shot falha nesse critério.

In [ ]:
zero_shot_prompt = (
    f"Contexto:\n{results[0].citation}: {results[0].text}\n\n"
    f"{results[1].citation}: {results[1].text}\n\n"
    f"Pergunta: {QUESTION}\nResponda com base apenas no contexto acima."
)
raw_zero_shot = llm_prose.generate(zero_shot_prompt) 
print("--- Saída bruta (zero-shot) ---")
print(raw_zero_shot)

--- Saída bruta (zero-shot) ---
A pena prevista para o homicídio (matar alguém) não está explicitamente mencionada nas normas apresentadas, mas é possível inferir que a pena seria de reclusão por um período que varia de acordo com as circunstâncias do crime.


In [4]:
parsed_zero_shot = parse_answer(raw_zero_shot)
print(f"parse_ok:   {parsed_zero_shot.parse_ok}")
print(f"abstained:  {parsed_zero_shot.abstained}")
print(f"citations:  {parsed_zero_shot.citations}")
print(f"confidence: {parsed_zero_shot.confidence}")

parse_ok:   False
abstained:  True
citations:  []
confidence: 0.0


**Leitura:** o modelo respondeu em prosa livre. `parse_answer` não encontra um objeto JSON no
texto, então falha: `parse_ok=False`, `abstained=True`, `citations=[]`. Ainda que o modelo "saiba a resposta", o pipeline não pode extrair `answer`/
`citations` de forma confiável.

## Técnica 2 — JSON não-guiado (`format="json"`, sem contrato de chaves)

A segunda iteração adiciona `format="json"` mas **sem** fixar o contrato exato de chaves
(`answer`/`citations`/`abstained`/...) nem listar os ids citáveis.

**Critério de qualidade:** mesmo teste — `citations` precisa vir preenchido com os ids
exatos (`"CP:art121"`). A saída **é** JSON, mas o schema não é válido.

In [5]:
weak_system = (
    "Você responde perguntas de direito penal brasileiro com base no contexto fornecido. "
    "Responda em formato JSON."
)
weak_user = (
    f"Contexto:\n{results[0].citation}: {results[0].text}\n\n"
    f"{results[1].citation}: {results[1].text}\n\n"
    f"Pergunta: {QUESTION}"
)
raw_weak_json = llm.generate(weak_user, system=weak_system, format="json")
print("--- Saída bruta (JSON não-guiado) ---")
print(raw_weak_json)

--- Saída bruta (JSON não-guiado) ---
{ "resposta": "A pena para quem mata alguém varia dependendo da circunstância do crime e da intenção do agente, mas geralmente é classificada como homicídio doloso. A pena por homicídio doloso pode variar de 12 a 30 anos de reclusão, conforme o Código Penal Brasileiro." }


In [6]:
parsed_weak = parse_answer(raw_weak_json)
print(f"parse_ok:   {parsed_weak.parse_ok}")
print(f"abstained:  {parsed_weak.abstained}")
print(f"citations:  {parsed_weak.citations}")
print(f"confidence: {parsed_weak.confidence}")

parse_ok:   True
abstained:  False
citations:  []
confidence: 0.0


**Leitura:** `format="json"` garante que a saída seja um JSON válido, então
`parse_answer` consegue extrair o objeto (`parse_ok=True`). Mas sem o contrato
de chaves fixado no system prompt, o modelo é livre para nomear as chaves como quiser
(`"resposta"`, `"fonte"`, `"dispositivos"`, ...) em vez de exatamente `"answer"`/
`"citations"`/`"abstained"`.

## Técnica 3 — Few-shot + saída controlada por schema

- **Few-shot / *example-guided*:** o `SYSTEM_PROMPT` embute um **exemplo** de pergunta e resposta em JSON.
- **Saída restrita por schema:** a chamada usa `format=<json schema>` do Ollama (não apenas
  `"json"`), que restringe a geração a um formato exato.
- **Ids citáveis explícitos:** `build_user_prompt` lista os ids exatos disponíveis para
  citação (`"Ids disponíveis para citação: CP:art121, CP:art155"`), removendo qualquer
  ambiguidade sobre como o modelo deve referenciar cada dispositivo.

In [7]:
print(SYSTEM_PROMPT)

Você é um assistente que explica Direito Penal brasileiro a partir de trechos de normas fornecidos abaixo. Isto é uma explicação informativa, NÃO é aconselhamento jurídico.

Regras obrigatórias:
1. Responda SOMENTE com base nas PROVISÕES fornecidas no contexto. Não use conhecimento externo nem invente dispositivos.
2. Se as provisões fornecidas respondem à pergunta, RESPONDA (defina "abstained": false) e liste em "citations" as tags dos dispositivos que você usou, usando o id exato de cada provisão (por exemplo "CP:art121", sem colchetes).
3. Abstenha-se ("abstained": true) SOMENTE quando as provisões não permitirem responder; nesse caso explique brevemente por quê.
4. Responda em português do Brasil.
5. Responda ESTRITAMENTE em JSON, sem texto fora do objeto JSON, com exatamente estas chaves: "answer" (string), "citations" (lista de ids como "CP:art121"), "hierarchy_notes" (string, pode ser vazia), "abstained" (booleano), "confidence" (número entre 0 e 1).

EXEMPLO de resposta correta

In [8]:
user_prompt = build_user_prompt(QUESTION, results)
print(user_prompt)

PROVISÕES:

[CP:art121] (CP art. 121, situação: alterado)
Matar alguem:
Pena - reclusão, de seis a vinte anos.
Caso de diminuição de pena
§
1º Se o agente comete o crime impelido por motivo de relevante valor social ou moral, ou
sob o domínio de violenta emoção, logo em seguida a injusta provocação da vítima, o
juiz pode reduzir a pena de um sexto a um terço.
Homicídio qualificado
§
2° Se o homicídio é cometido:
I -
mediante paga ou promessa de recompensa, ou por outro motivo torpe;
II
- por motivo futil;
III
- com emprego de veneno, fogo, explosivo, asfixia, tortura ou outro meio insidioso ou
cruel, ou de que possa resultar perigo comum;
IV
- à traição, de emboscada, ou mediante dissimulação ou outro recurso que dificulte ou
torne impossivel a defesa do ofendido;
V - para assegurar a execução, a ocultação, a impunidade ou vantagem de outro
crime:
Pena - reclusão, de doze a
trinta anos.
Feminicídio
(Incluído pela Lei nº
13.104, de 2015)
VI -
(Revogado pela Lei nº
14.994, de 2024)
VII 

In [ ]:
# The same JSON schema used internally by direito_dados.generation.rag.answer_question
# (reproduced here to make the output contract that constrains generation visible).
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "citations": {"type": "array", "items": {"type": "string"}},
        "hierarchy_notes": {"type": "string"},
        "abstained": {"type": "boolean"},
        "confidence": {"type": "number"},
    },
    "required": ["answer", "citations", "abstained", "confidence"],
}

for attempt in range(1, 4):
    raw_schema = llm.generate(user_prompt, system=SYSTEM_PROMPT, format=ANSWER_SCHEMA)
    parsed_schema = parse_answer(raw_schema)
    print(f"--- tentativa {attempt} ---")
    print(raw_schema)
    if not parsed_schema.abstained and "CP:art121" in parsed_schema.citations:
        break

In [10]:
print(f"parse_ok:   {parsed_schema.parse_ok}")
print(f"abstained:  {parsed_schema.abstained}")
print(f"citations:  {parsed_schema.citations}")
print(f"confidence: {parsed_schema.confidence}")
print(f"answer:     {parsed_schema.answer}")

parse_ok:   True
abstained:  False
citations:  ['CP:art121', 'CP:art155']
confidence: 1.0
answer:     A pena para quem mata alguém varia dependendo do contexto e da intenção, mas em geral é considerado homicídio e pode ter penas de prisão variando de alguns anos a até mesmo pena de morte em alguns países.


**Leitura (resultado real acima):** o schema explícito resolve o problema de **forma**: a
saída é sempre um JSON com exatamente as chaves esperadas, `parse_ok=True`, e
`abstained=False` porque o contexto de fato contém a resposta. Mas o schema **não** garante
conteúdo correto. Na execução capturada acima, o modelo cita `["CP:art121", "CP:art155"]` —
incluindo o art. 155 (furto), que não tem relação com uma pergunta sobre homicídio — e a
resposta menciona "pena de morte em alguns países", uma afirmação que não está no contexto
fornecido e que também não corresponde à legislação brasileira (não há pena de morte para
homicídio no Brasil). Este é, portanto, um exemplo de resposta **fraca/incorreta no
conteúdo**, mesmo com o formato perfeito: o contrato de saída resolve "a resposta vem no
formato certo?", mas não resolve "a resposta está certa?" — são dois problemas
independentes. Citações erradas como esta não seriam pegas só checando `parse_ok`/
`abstained`; um sistema de produção precisa de uma verificação adicional de que as
citações realmente sustentam o texto da resposta.

### Abstenção coerente: mesma técnica, contexto insuficiente

O contrato de saída não deveria fazer o modelo "inventar" `abstained=false` quando não há
base. Testamos a mesma técnica 3 (few-shot + schema) mas com contexto vazio —
`build_user_prompt` gera, nesse caso, uma instrução explícita para abster-se.

In [11]:
empty_prompt = build_user_prompt("Qual o prazo prescricional do IPTU?", [])
print(empty_prompt)

PROVISÕES: nenhuma provisão relevante foi encontrada na base de normas para esta pergunta. Não há contexto para responder.
Responda com "abstained": true, explicando que não há base normativa recuperada.

PERGUNTA: Qual o prazo prescricional do IPTU?


In [12]:
raw_abstain = llm.generate(empty_prompt, system=SYSTEM_PROMPT, format=ANSWER_SCHEMA)
print("--- Saída bruta (schema, contexto vazio) ---")
print(raw_abstain)

parsed_abstain = parse_answer(raw_abstain)
print(f"\nparse_ok:   {parsed_abstain.parse_ok}")
print(f"abstained:  {parsed_abstain.abstained}")
print(f"citations:  {parsed_abstain.citations}")

--- Saída bruta (schema, contexto vazio) ---
{"answer": "", "citations": [], "abstained": true, "confidence": 1.0}

parse_ok:   True
abstained:  True
citations:  []


**Leitura:** sem provisões no contexto, o modelo define `abstained=true` e não inventa
`citations` — o critério de qualidade (citações corretas *e* abstenção coerente) é
satisfeito nos dois sentidos: cita quando pode, abstém-se quando não pode.

## Técnica 4 — Chain-of-thought estruturado (raciocínio antes do veredito)

As três técnicas anteriores lidam com a **forma** da resposta (a saída é parseável? as chaves
estão certas?). Chain-of-thought lida com o problema do **raciocínio**. 

A resolução de **conflito aparente de normas** exige um raciocínio em etapas (identificar dispositivos aplicáveis → testar a relação de especialidade → concluir
pela *lex specialis*).

O experimento usa um caso clássico da doutrina: **homicídio (art. 121, norma geral) vs.
infanticídio (art. 123, norma especial)**, em duas variantes sobre o mesmo contexto:

- **Variante A (veredito direto):** o schema pede apenas `dispositivo` + `justificativa`.
- **Variante B (CoT estruturado):** o schema inclui um campo `raciocinio` (lista de passos)
  **antes** dos campos de veredito. A ordem importa: como a geração é autoregressiva, os
  tokens de raciocínio são gerados *antes* do veredito. O veredito é condicionado ao
  raciocínio.

In [ ]:
# Context: two provisions in a specialty tension — homicide (art. 121, general rule) and
# infanticide (art. 123, special rule).
art123 = next(a for a in cp.articles if a.number == "123")

CASE = (
    "Uma mãe, sob a influência do estado puerperal, mata o próprio filho durante o parto. "
    "Qual dispositivo do Código Penal se aplica a essa conduta?"
)

conflict_context = (
    f"[CP:art121] (art. 121 — {art121.status.value})\n{art121.text[:400]}\n\n"
    f"[CP:art123] (art. 123 — {art123.status.value})\n{art123.text[:400]}"
)

# --- Variant A: direct verdict (no reasoning instruction) ---
DIRECT_SCHEMA = {
    "type": "object",
    "properties": {
        "dispositivo": {"type": "string"},
        "justificativa": {"type": "string"},
    },
    "required": ["dispositivo", "justificativa"],
}
system_direct = (
    "Você é um assistente de direito penal brasileiro. Com base SOMENTE nos dispositivos "
    "fornecidos, responda qual se aplica ao caso. Responda em JSON com as chaves "
    '"dispositivo" (o id, ex.: "CP:art121") e "justificativa" (uma frase).'
)
raw_direct = llm.generate(f"{conflict_context}\n\nCASO: {CASE}", system=system_direct,
                          format=DIRECT_SCHEMA)
print("=== Variante A — veredito direto ===")
print(raw_direct)

# --- Variant B: structured chain-of-thought (reasoning BEFORE the verdict) ---
COT_SCHEMA = {
    "type": "object",
    "properties": {
        "raciocinio": {"type": "array", "items": {"type": "string"}},
        "dispositivo": {"type": "string"},
        "justificativa": {"type": "string"},
    },
    "required": ["raciocinio", "dispositivo", "justificativa"],
}
system_cot = (
    "Você é um assistente de direito penal brasileiro. Com base SOMENTE nos dispositivos "
    "fornecidos, RACIOCINE PASSO A PASSO antes de concluir: em \"raciocinio\", liste os "
    "passos — (1) quais dispositivos descrevem a conduta; (2) se há relação de "
    "especialidade entre eles (a norma especial contém todos os elementos da geral, mais "
    "um elemento especializante); (3) qual prevalece e por quê (lex specialis). Só então "
    'preencha "dispositivo" (o id) e "justificativa".'
)
raw_cot = llm.generate(f"{conflict_context}\n\nCASO: {CASE}", system=system_cot,
                       format=COT_SCHEMA)
print()
print("=== Variante B — chain-of-thought estruturado ===")
print(raw_cot)

**Leitura (resultado real acima):** as duas variantes acertam o veredito (`CP:art123`,
infanticídio) — o caso é clássico o bastante para o modelo de 8B resolver mesmo sem ajuda.
A diferença está no que cada saída **permite auditar**: a Variante A entrega uma
justificativa de uma frase; a Variante B expõe o teste de especialidade passo a passo —
"(2) ... o art. 123 especifica que a conduta deve ser cometida pela gestante sob influência
do estado puerperal e contra o próprio filho" → "(3) o dispositivo específico prevalece"
(*lex specialis*). Em um domínio de alto risco como o jurídico, em que o sistema só produz
**candidatos para revisão humana**, essa auditabilidade é o valor real do chain-of-thought:
o revisor confere o *porquê*, não apenas o *o quê*. É por isso que o adjudicador de
antinomias de produção (`direito_dados.conflicts.detector`) exige o campo `rationale` no
schema do veredito.

## Resumo da iteração

| Técnica | Mecanismo | `parse_ok` | `citations` preenchido corretamente? | `abstained` coerente? |
|---|---|---|---|---|
| 1. Zero-shot | instrução livre, sem `format` | não | não (nem chega a parsear) | não |
| 2. JSON não-guiado | `format="json"`, sem contrato de chaves | sim | não confiável (chaves divergentes) | não confiável |
| 3. Few-shot + schema | `SYSTEM_PROMPT` (exemplo) + `format=<schema>` + ids explícitos | sim | sim | sim |

A Técnica 4 (chain-of-thought estruturado) é ortogonal às três primeiras: estas resolvem a
**forma** da saída; aquela permite a um revisor verificar o caminho até o veredito.

Esse é o caminho que `direito_dados.generation.rag.answer_question` percorre: constrói o prompt com `build_user_prompt` (listando os ids citáveis), chama o
modelo com `system=SYSTEM_PROMPT` e `format=<schema idêntico ao ANSWER_SCHEMA acima>`, e
passa a saída por `parse_answer`.